# V2a 220119_F2_run11: Shared Manifold-Transition Comparison

Canonical adopted analysis: one raw training-owned manifold and one held-out event table serve every estimator. Run the synthetic framework validation before interpreting biological results.
Prerequisite: this recording's `00_preprocessing/preprocess.ipynb` raw_recording checkpoint. Individual graph-state, community, manifold and linking notebooks are optional.
The helper rebuilds identity-preprocessed windows from raw_recording and disables per-window standardization; it never reuses descriptive manifolds or globally standardized windows.
All eight retained estimators are requested by default. Fixed settings are development choices; unavailable fits and insufficient raw support remain explicit.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
from dataclasses import replace

from omegaconf import OmegaConf

from effectome.data_module import WindowConfig
from effectome.workflows.cross_method import load_cross_method_profile, write_cross_method_report
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_shared_transition_comparison,
    stage_artifact_path,
)

DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220119_F2_run11"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RUN_ID = "shared-transitions"
METHOD = "cross-method"
FORCE = False

shared_run = make_run(OUTPUT_ROOT, "reference", "shared")
RAW_RECORDING_PATH = stage_artifact_path(shared_run, "raw_recording")
WINDOW_CONFIG_PATH = PROJECT_ROOT / "conf" / "windowing" / "primary_physical.yaml"
window_data = OmegaConf.to_container(OmegaConf.load(WINDOW_CONFIG_PATH), resolve=True)
if not isinstance(window_data, dict):
    raise TypeError(f"Expected a mapping in {WINDOW_CONFIG_PATH}")
WINDOW_CFG = replace(WindowConfig(**window_data), standardize_per_window=False)
COMPARISON_CFG, METHOD_CONFIGS = load_cross_method_profile("development")
GROUPS = {}  # Optional anatomy/training-established groups: {"name": ["neuron_id", ...]}

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title="V2a 220119_F2_run11 shared manifold-transition comparison",
    expected_stages=["raw_recording_input", "cross_method_windows", "cross_method_comparison"],
    force=FORCE,
)


In [ ]:
report = run_shared_transition_comparison(
    run,
    raw_recording_path=RAW_RECORDING_PATH,
    window_cfg=WINDOW_CFG,
    comparison_cfg=COMPARISON_CFG,
    method_configs=METHOD_CONFIGS,
    groups=GROUPS,
    force=FORCE,
)
print("Shared fitted reference:", report["fit_scope_id"])
print("Training/evaluation events:",
      len(report["events"]["train_rows"]), len(report["events"]["evaluation_rows"]))
for method, result in report["methods"].items():
    if result["status"] == "available":
        print(method, "held-out graph gain:", result["prediction"]["log_loss_gain"])
        print("  Native graph states:", result["native_graph_states"]["status"])
    else:
        print(method, "unavailable:", result["reason"])
print("Neuronal identity source:", report["neuron_id_source"])
print_stage_status(run)


## Paired reports and candidate interpretation

The empirical manifold transition matrix is shared. Each method contributes predicted conditional transitions and, where available, its own graph-state correspondence. Positive omission scores are transition-specific predictive evidence, not biological necessity. Inspect stays, competing destinations, unavailable fits, and synthetic recovery before selecting biological candidates.


In [ ]:
REPORT_DIRECTORY = run.run_dir / "reports"
paths = write_cross_method_report(report, REPORT_DIRECTORY)
print("Comparison, candidate table, event table and summary:")
for path in paths:
    print(path)
print("Family-aware candidate summaries:", len(report["family_rank_synthesis"]))
print("Claim boundaries:", report["scientific_claims"])
